Correlation

In [1]:
import pandas as pd

In [2]:
df = pd.DataFrame({"Age":[22,25,30,35],
                   "Salary":[30000,40000,50000,60000],
                   "Purchased":[0,0,1,1]})
df

,Age,Salary,Purchased
0,22,30000,0
1,25,40000,0
2,30,50000,1
3,35,60000,1


In [6]:
corr = df.corr()

target = 'Purchased'

corr[target]

Age          0.909137
Salary       0.894427
Purchased    1.000000
Name: Purchased, dtype: float64

VARIANCE THRESHOLD   --> Remove Features Who's Values Hardly change

In [8]:
from sklearn.feature_selection import VarianceThreshold
import pandas as pd

In [9]:
df = pd.DataFrame({ "A":[1,1,1,1],
               "B":[1,2,3,4]})
df

,A,B
0,1,1
1,1,2
2,1,3
3,1,4


In [10]:
selector = VarianceThreshold()

new_df = selector.fit_transform(df)
new_df                              # Remove Features Who's Values Hardly change

array([[1],
       [2],
       [3],
       [4]])

RFE (Recursive Feature Elimination)

In [11]:
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

In [12]:
df = pd.DataFrame({
    "Age":[22,25,30,35],
    "Salary":[30000,40000,50000,60000],
    "Height":[170,175,180,185]
})

df

,Age,Salary,Height
0,22,30000,170
1,25,40000,175
2,30,50000,180
3,35,60000,185


In [13]:
y = [0,0,1,1]     # Target varaible for eg: Purchased (y) 1->Buyed, 0->Didn't Buyes  -> it contains 0s and 1s (looks like classification) we're using LinearRegression, which predicts continuous values. For learning RFE, this is acceptable, but for a real classification problem we'd typically use LogisticRegression
                  # "Which features help me predict the target?" Keep only helping Features
                  # For each row in df, this is the correct answer/output that the model should learn to predict.
model = LinearRegression()   # RFE needs a model that can assign importance (coefficients).

selector = RFE(model, n_features_to_select = 2)  # Linear Regression using in the RFE Feature selector to Rank the feature by Linearly Regressing the Features values.
                                                 # "Select the best 2 features." # RFE will repeatedly remove the least important feature until only 2 remain.
selector.fit(df, y)
selector.support_            # support_ is a Boolean array indicating whether each feature was selected.
                             # Age,Salary selected Height not selected  --> Height Feature has less importance understanded by RFE selector using linearRegr model 

array([ True,  True, False])

In [14]:
print(df.columns[selector.support_])
print(selector.ranking_)        # Rank 1 always means the feature was selected.Higher Rank removed

Index(['Age', 'Salary'], dtype='object')
[1 1 2]


In [15]:
help(RFE)

Help on class RFE in module sklearn.feature_selection._rfe:

class RFE(sklearn.feature_selection._base.SelectorMixin, sklearn.base.MetaEstimatorMixin, sklearn.base.BaseEstimator)
 |  RFE(
 |      estimator,
 |      *,
 |      n_features_to_select=None,
 |      step=1,
 |      verbose=0,
 |      importance_getter='auto'
 |  )
 |
 |  Feature ranking with recursive feature elimination.
 |
 |  Given an external estimator that assigns weights to features (e.g., the
 |  coefficients of a linear model), the goal of recursive feature elimination
 |  (RFE) is to select features by recursively considering smaller and smaller
 |  sets of features. First, the estimator is trained on the initial set of
 |  features and the importance of each feature is obtained either through
 |  any specific attribute or callable.
 |  Then, the least important features are pruned from current set of features.
 |  That procedure is recursively repeated on the pruned set until the desired
 |  number of features to s

STEPWISE SELECTION

In [16]:
import pandas as pd

df = pd.DataFrame({
    "Age":[22,25,30,35,40],
    "Salary":[30000,40000,50000,60000,70000],
    "Height":[170,175,180,185,190],
    "Experience":[1,3,5,7,9],
    "Target":[0,0,1,1,1]
})

df

,Age,Salary,Height,Experience,Target
0,22,30000,170,1,0
1,25,40000,175,3,0
2,30,50000,180,5,1
3,35,60000,185,7,1
4,40,70000,190,9,1


In [17]:
x = df[["Age","Salary","Height","Experience",]]
y = df["Target"] 

print(x)
y

   Age  Salary  Height  Experience
0   22   30000     170           1
1   25   40000     175           3
2   30   50000     180           5
3   35   60000     185           7
4   40   70000     190           9


0    0
1    0
2    1
3    1
4    1
Name: Target, dtype: int64

In [ ]:
import statsmodels.api as sm

selected_features = []
remaining_features = [x.columns]

while remaining_features:
    best_feature = None
    best_pvalue = 1

    for feature in remaining_features:
        features = selected_features + [feature]

        x_model = sm.add_constant(x[features])

        model = sm.OLS(y,x_model).fit()

        p_value = model.pvalues[feature]

In [21]:
# forward Stepwise selector using mlxtend library

In [30]:
df = pd.DataFrame({
    "Age":[22,25,30,35,40],
    "Salary":[30000,40000,50000,60000,70000],
    "Height":[170,175,180,185,190],
    "Experience":[1,3,5,7,9],
})   # features  --> Target shouldnt be used as feature that should be given in another variable to calculate using the selectors


y = [0,0,1,1,1]  # Target values

print(x)
print(y)

   Age  Salary  Height  Experience
0   22   30000     170           1
1   25   40000     175           3
2   30   50000     180           5
3   35   60000     185           7
4   40   70000     190           9
[0, 0, 1, 1, 1]


In [42]:
from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

model = LinearRegression()

stepwiseselector = SequentialFeatureSelector(model, k_features=2, forward=True, cv=2) # Model of data calculation, total how much feature needed at end, Forward/backward stepwise feature selection
                                                   # SequentialFeatureSelector uses 5-fold cross-validation by default so changed cv into 2.
stepwiseselector.fit(df,y)

print(stepwiseselector.k_feature_names_)
print(stepwiseselector.k_score_)  # R² (R-squared) score because of using LinearRegression -- R² tells us how well the model predicts the target.

('Age', 'Salary')
-0.9999999999999998


In [2]:
help(SequentialFeatureSelector)

Help on class SequentialFeatureSelector in module mlxtend.feature_selection.sequential_feature_selector:

class SequentialFeatureSelector(mlxtend.utils.base_compostion._BaseXComposition, sklearn.base.MetaEstimatorMixin)
 |  SequentialFeatureSelector(
 |      estimator,
 |      k_features=1,
 |      forward=True,
 |      floating=False,
 |      verbose=0,
 |      scoring=None,
 |      cv=5,
 |      n_jobs=1,
 |      pre_dispatch='2*n_jobs',
 |      clone_estimator=True,
 |      fixed_features=None,
 |      feature_groups=None,
 |      tol=None
 |  )
 |
 |  Sequential Feature Selection for Classification and Regression.
 |
 |  Parameters
 |  ----------
 |  estimator : scikit-learn classifier or regressor
 |  k_features : int or tuple or str (default: 1)
 |      Number of features to select,
 |      where k_features < the full feature set.
 |      New in 0.4.2: A tuple containing a min and max value can be provided,
 |          and the SFS will consider return any feature combination betw

In [38]:
import mlxtend
print(mlxtend.__version__)

0.25.0
